# Stagnation detection in practice

## Recursive summation

We show how, for a simple program, code can be modified to perform stagnation detection

### Approximation of $\pi$

A relatively simple expression for $\pi$ in the form of a series is given by

$$ \pi / 4 = \sum_{i = 0}^{\infty} \frac{(-1)^i}{2i + 1} = 1 - \frac{1}{3} + \frac{1}{5} - \frac{1}{7} + ...$$

We can evaluate this in Julia using the following code


In [1]:
n = 20000000

acc = 1.0
for i in 1:(n-1)
    acc += ((-1)^i)/(2*i + 1)
end

println(4*acc)

3.141592603589817


This evaluates $n$ terms in the series and performs $n-1$ additions.
To perform stagnation detection, we modify the code to use fixed data types that override addition to record everything going on.

In [2]:
using StagnationDetection

n = 20000000

acc_stag::Float64SD = 1.0
for i in 1:(n-1)
    acc_stag += Float64((-1)^i)/(2*i + 1)
end

println(4*acc_stag)

3.141592603589817


We can change the format to any of `Float64SD`, `Float32SD` or `Float16SD`. However, type promotions behave like normal, so the other terms in the operations must be the correct format. We can perform the same operation in single precision.

In [3]:
n = 20000000

acc_stag2::Float32SD = 1.0
for i in 1:(n-1)
    acc_stag2 += Float32((-1)^i)/(2*i + 1)
end

println(4*acc_stag2)

3.1415968f0


A `FloatSD` variable has fields which record data regarding stagnation. We can access these directly, or we can use `report` to generate a message about stagnation.

In [4]:
report(acc_stag2)


--------------- SUMMARY ----------------
Floating point additions performed: 19999999.
Total stagnation of length 3222783.
The condition for total is u = 0 and for the smaller addend to be rounded off completely in a given addition.
See paper: Analysis and Detection of Stagnation in Floating-Point Summation,
Mantas Mikaitis and William Woolfenden, 2026 (in progress)

-------- PROCESSING OF RESULTS ---------
u is the total number of bits which were not shifted off in the addition.
Total number of bits processed u = 16777191.
Average uᵢ = 0.8388595919429795.

------------ END OF SUMMARY ------------


The result tells us that after roughly $2 \times 10^7$ additions in single precision, total stagnation occurs and the quality of the approximation, which we can see is accurate to $5$ decimal places, will not improve.